In [1]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

PARQUET_BASE = "hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease"
dataset = load_dataset("parquet", data_files={
    "train": f"{PARQUET_BASE}/train/0000.parquet",
    "validation": f"{PARQUET_BASE}/validation/0000.parquet",
    "test": f"{PARQUET_BASE}/test/0000.parquet",
})

label_names = ["O", "B-Disease", "I-Disease"]
num_labels = len(label_names)

# Note: using bert-base-cased NOT biobert
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        max_length=512,
        truncation=True
    )
    all_aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        aligned_labels = []
        previous_word_id = None
        for word_id in tokenized_inputs.word_ids(batch_index=i):
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            previous_word_id = word_id
        all_aligned_labels.append(aligned_labels)
    tokenized_inputs["labels"] = all_aligned_labels
    return tokenized_inputs

# Preprocess with general BERT tokenizer
import functools
tokenized_bert = dataset.map(
    functools.partial(tokenize_and_align, tokenizer=tokenizer_bert),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Dataset preprocessed with general BERT tokenizer!")
print(tokenized_bert)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0000.parquet:   0%|          | 0.00/425k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/74.7k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/77.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

Dataset preprocessed with general BERT tokenizer!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})


In [3]:
from transformers import AutoModelForTokenClassification

model_bert = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels,
    id2label={0: "O", 1: "B-Disease", 2: "I-Disease"},
    label2id={"O": 0, "B-Disease": 1, "I-Disease": 2}
)
print("General BERT loaded!")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

General BERT loaded!


In [4]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score
from google.colab import drive
drive.mount('/content/drive')

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer_bert, padding=True
)

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels, true_predictions = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_label_seq, true_pred_seq = [], []
        for pred, label in zip(pred_seq, label_seq):
            if label != -100:
                true_label_seq.append(label_names[label])
                true_pred_seq.append(label_names[pred])
        true_labels.append(true_label_seq)
        true_predictions.append(true_pred_seq)
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/bert-base-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    logging_steps=50,
    fp16=True
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=tokenized_bert["train"],
    eval_dataset=tokenized_bert["validation"],
    processing_class=tokenizer_bert,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Training general BERT baseline...")
trainer_bert.train()

Mounted at /content/drive
Training general BERT baseline...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.053200,0.057297,0.727797,0.801779,0.762999
2,0.033560,0.052410,0.763429,0.848793,0.803851
3,0.018587,0.058473,0.778291,0.856417,0.815487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1020, training_loss=0.053245919358496575, metrics={'train_runtime': 494.5896, 'train_samples_per_second': 32.955, 'train_steps_per_second': 2.062, 'total_flos': 4258914342276096.0, 'train_loss': 0.053245919358496575, 'epoch': 3.0})

In [5]:
bert_results = trainer_bert.evaluate(tokenized_bert["test"])
print("=== General BERT Results ===")
print(f"Precision: {bert_results['eval_precision']:.4f}")
print(f"Recall:    {bert_results['eval_recall']:.4f}")
print(f"F1:        {bert_results['eval_f1']:.4f}")

=== General BERT Results ===
Precision: 0.7975
Recall:    0.8781
F1:        0.8359


In [6]:
print("=" * 50)
print("=== ABLATION STUDY RESULTS ===")
print("=" * 50)
print(f"{'Model':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 50)
print(f"{'bert-base-cased':<25} {bert_results['eval_precision']:>10.4f} {bert_results['eval_recall']:>10.4f} {bert_results['eval_f1']:>10.4f}")
print(f"{'BioBERT (ours)':<25} {'0.8479':>10} {'0.9000':>10} {'0.8732':>10}")
print("=" * 50)
improvement = 0.8732 - bert_results['eval_f1']
print(f"\nBioBERT improvement over BERT: +{improvement:.4f} F1")

=== ABLATION STUDY RESULTS ===
Model                      Precision     Recall         F1
--------------------------------------------------
bert-base-cased               0.7975     0.8781     0.8359
BioBERT (ours)                0.8479     0.9000     0.8732

BioBERT improvement over BERT: +0.0373 F1
